# Exercise 2 – Lactic Acid Inhibition of *Streptococcus cremoris*

This notebook analyses the inhibitory effect of lactic acid on the growth of *S. cremoris*, based on data from Bibal et al. (1988, 1989).  
The four sub-questions are treated sequentially:

- **(a)** Convert total lactic acid concentration → undissociated acid concentration P, then plot the relative specific growth rate µ/µ₀ vs P  
- **(b)** Fit model 1 (Monod + competitive-like inhibition) to find K_I using `polyfit`  
- **(c)** Fit model 2 (Monod + linear inhibition term) and compare both models quantitatively  
- **(d)** Plot µ_rel vs pH for two total lactic acid concentrations and find the pH at which growth stops

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# -- Matplotlib style --
plt.rcParams.update({'figure.dpi': 120, 'font.size': 12})

---
## Part (a) – Relative growth rate vs undissociated acid concentration P

### Theory
Lactic acid is a weak acid that partially dissociates in solution:
$$\text{HA} \rightleftharpoons \text{H}^+ + \text{A}^-$$

The Henderson–Hasselbalch equation gives the fraction of acid that is **undissociated**:
$$f_{\text{HA}} = \frac{1}{1 + 10^{\text{pH} - \text{pK}_a}}$$

The undissociated acid concentration (in mM) is therefore:
$$P = f_{\text{HA}} \times \frac{p_{\text{total}}}{M_w} \times 1000$$

where $p_{\text{total}}$ is the total lactic acid concentration in g/L and $M_w = 90.08$ g/mol.

The experiment was conducted at **pH = 6.3**, pKₐ = 3.88.

In [ ]:
# ── Experimental data ──────────────────────────────────────────────────────────
# Total lactic acid concentration p_total (g/L) and specific growth rate mu (h⁻¹)
p_total_gL = np.array([0.0, 12.0, 39.0, 55.0])   # g/L
mu_data    = np.array([0.90, 0.68, 0.52, 0.13])   # h⁻¹

# ── Physical / chemical constants ──────────────────────────────────────────────
pH   = 6.3
pKa  = 3.88
Mw   = 90.08   # g/mol, molar mass of lactic acid

# ── Henderson–Hasselbalch: fraction of undissociated acid ──────────────────────
# f_HA = 1 / (1 + 10^(pH - pKa))
f_HA = 1.0 / (1.0 + 10**(pH - pKa))
print(f"Fraction undissociated (f_HA) at pH {pH}: {f_HA:.6f}")

# ── Convert total concentration (g/L) → undissociated acid (mM) ───────────────
# P [mM] = f_HA * (p_total [g/L] / Mw [g/mol]) * 1000 [mmol/mol]
P_mM = f_HA * (p_total_gL / Mw) * 1000.0
print("\nP_total (g/L) → P undissociated (mM):")
for pt, P in zip(p_total_gL, P_mM):
    print(f"  {pt:5.1f} g/L  →  {P:.4f} mM")

# ── Relative specific growth rate ─────────────────────────────────────────────
mu0     = mu_data[0]        # µ(P=0) = µmax under substrate saturation
mu_rel  = mu_data / mu0

print(f"\nµ(P=0) = µmax = {mu0} h⁻¹")

In [ ]:
# ── Plot µ_rel vs P ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(P_mM, mu_rel, 'ko', markersize=8, label='Experimental data')
ax.set_xlabel('Undissociated lactic acid P (mM)')
ax.set_ylabel(r'Relative growth rate  $\mu / \mu_0$  (–)')
ax.set_title('Part (a) – Relative growth rate vs undissociated acid P')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('part_a_mu_rel_vs_P.png')
plt.show()

---
## Part (b) – Model 1: finding K_I with `polyfit`

### Why µ(P=0) = µ_max?
The data are collected **under substrate saturation**, so S ≫ K_S, meaning:
$$\frac{S}{K_S + S} \approx 1$$
The growth equation therefore reduces to:
$$\mu = \mu_{\max} \cdot 1 \cdot \frac{1}{1 + P/K_I} \xrightarrow{P \to 0} \mu_{\max}$$

### Model 1 equation
Under substrate saturation the model becomes:
$$\mu = \frac{\mu_{\max}}{1 + P/K_I}$$

Rearranging to a **linear** form suitable for `polyfit`:
$$\frac{1}{\mu} = \frac{1}{\mu_{\max}} + \frac{1}{\mu_{\max} K_I} \cdot P$$

Plotting $1/\mu$ vs $P$ gives a straight line with:
- intercept = $1/\mu_{\max}$
- slope = $1/(\mu_{\max} K_I)$  →  $K_I = 1 / (\text{slope} \times \mu_{\max})$

In [ ]:
# ── Linearised fit using polyfit ───────────────────────────────────────────────
# y = 1/mu,  x = P
inv_mu = 1.0 / mu_data

# degree-1 polynomial: coefficients = [slope, intercept]
coeffs = np.polyfit(P_mM, inv_mu, 1)
slope_b, intercept_b = coeffs

mu_max_b = 1.0 / intercept_b
KI_b     = 1.0 / (slope_b * mu_max_b)

print(f"Linear fit:  slope = {slope_b:.6f} h/mM,  intercept = {intercept_b:.6f} h")
print(f"µ_max (from intercept) = {mu_max_b:.4f} h⁻¹")
print(f"K_I  (Model 1)         = {KI_b:.4f} mM")

# ── Model 1 predictions ───────────────────────────────────────────────────────
P_fit = np.linspace(0, max(P_mM) * 1.05, 300)
mu_model1 = mu_max_b / (1.0 + P_fit / KI_b)
mu_rel_model1 = mu_model1 / mu_max_b

# Also compute predictions at experimental P for residuals
mu_pred1_data = mu_max_b / (1.0 + P_mM / KI_b)

In [ ]:
# ── Plot linearisation ─────────────────────────────────────────────────────────
P_line = np.linspace(0, max(P_mM), 200)
inv_mu_fit = np.polyval(coeffs, P_line)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(P_mM, inv_mu, 'ko', markersize=8, label='1/µ data')
ax.plot(P_line, inv_mu_fit, 'b-', label=f'Linear fit (slope={slope_b:.4f}, intercept={intercept_b:.4f})')
ax.set_xlabel('Undissociated lactic acid P (mM)')
ax.set_ylabel(r'$1/\mu$ (h)')
ax.set_title('Part (b) – Linearised plot for Model 1 (polyfit)')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('part_b_linear_fit.png')
plt.show()

---
## Part (c) – Model 2: linear inhibition term

### Model 2 equation
Under substrate saturation:
$$\mu = \mu_{\max} \left(1 - \frac{P}{P_{\max}}\right)$$

The single parameter to estimate is **P_max** (the concentration of undissociated acid at which growth stops completely).  
Re-written in relative terms:
$$\frac{\mu}{\mu_{\max}} = 1 - \frac{P}{P_{\max}}$$

This is a straight line in µ vs P, so `polyfit(P, mu, 1)` works directly.

### Model comparison – quantitative measure
We use the **Root Mean Square Error (RMSE)** on the original µ values:
$$\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(\mu_i^{\text{data}} - \mu_i^{\text{model}})^2}$$

and the **coefficient of determination R²**:
$$R^2 = 1 - \frac{\sum(\mu_i^{\text{data}} - \mu_i^{\text{model}})^2}{\sum(\mu_i^{\text{data}} - \bar{\mu})^2}$$

In [ ]:
# ── Model 2: linear fit  µ = µ_max*(1 - P/P_max) ─────────────────────────────
# Fit: mu vs P is linear → polyfit gives [slope2, intercept2]
# slope2 = -µ_max / P_max,  intercept2 = µ_max
coeffs2 = np.polyfit(P_mM, mu_data, 1)
slope2, intercept2 = coeffs2

mu_max_c = intercept2               # µ_max from linear fit
P_max_c  = -mu_max_c / slope2       # P at which µ = 0

print(f"Model 2 – linear fit on µ vs P:")
print(f"  slope     = {slope2:.6f} h⁻¹/mM")
print(f"  intercept = {intercept2:.4f} h⁻¹  (= µ_max estimate)")
print(f"  P_max     = {P_max_c:.4f} mM")

# ── Model 2 predictions ───────────────────────────────────────────────────────
mu_model2     = mu_max_c * (1.0 - P_fit / P_max_c)
mu_rel_model2 = mu_model2 / mu_max_c
mu_pred2_data = np.polyval(coeffs2, P_mM)

# ── Helper: R² and RMSE ───────────────────────────────────────────────────────
def r2_rmse(y_data, y_pred):
    SS_res = np.sum((y_data - y_pred)**2)
    SS_tot = np.sum((y_data - np.mean(y_data))**2)
    r2   = 1.0 - SS_res / SS_tot
    rmse = np.sqrt(np.mean((y_data - y_pred)**2))
    return r2, rmse

r2_m1, rmse_m1 = r2_rmse(mu_data, mu_pred1_data)
r2_m2, rmse_m2 = r2_rmse(mu_data, mu_pred2_data)

print(f"\nModel 1 (hyperbolic):  R² = {r2_m1:.5f},  RMSE = {rmse_m1:.5f} h⁻¹")
print(f"Model 2 (linear):      R² = {r2_m2:.5f},  RMSE = {rmse_m2:.5f} h⁻¹")

In [ ]:
# ── Combined plot: both models + data ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(P_mM, mu_rel, 'ko', markersize=9, zorder=5, label='Experimental data')
ax.plot(P_fit, mu_rel_model1, 'b-',  lw=2,
        label=f'Model 1 – hyperbolic  (K_I={KI_b:.2f} mM, R²={r2_m1:.4f})')
ax.plot(P_fit, mu_rel_model2, 'r--', lw=2,
        label=f'Model 2 – linear      (P_max={P_max_c:.2f} mM, R²={r2_m2:.4f})')

ax.axhline(0, color='k', lw=0.8, linestyle=':')
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
ax.set_xlabel('Undissociated lactic acid P (mM)')
ax.set_ylabel(r'Relative growth rate  $\mu / \mu_0$  (–)')
ax.set_title('Part (c) – Comparison of inhibition models')
ax.legend(fontsize=10)
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('part_c_model_comparison.png')
plt.show()

print("\n→ The model with the higher R² (lower RMSE) fits the data better.")

---
## Part (d) – Growth rate vs pH at fixed total lactic acid concentrations

### Strategy
Using the **better model** identified in part (c), we write µ_rel as a function of pH:
$$P(\text{pH}) = f_{\text{HA}}(\text{pH}) \times \frac{p_{\text{total}}}{M_w} \times 1000$$

with $f_{\text{HA}}(\text{pH}) = \dfrac{1}{1 + 10^{\text{pH} - \text{pK}_a}}$.

We then plot µ_rel vs pH for:
- $p_{\text{total}} = 1$ g/L
- $p_{\text{total}} = 10$ g/L

Growth stops when µ_rel = 0, i.e. when $P = P_{\max}$ (Model 2) or equivalently $\mu \leq 0$.

In [ ]:
# ── Select the better model ───────────────────────────────────────────────────
# We compare by R²: pick the model with higher R²
if r2_m1 >= r2_m2:
    best_model = 1
    print(f"Best model: Model 1 (hyperbolic), K_I = {KI_b:.4f} mM")
    def mu_rel_of_P(P):
        return 1.0 / (1.0 + P / KI_b)  # already relative (µ_max cancels)
else:
    best_model = 2
    print(f"Best model: Model 2 (linear), P_max = {P_max_c:.4f} mM")
    def mu_rel_of_P(P):
        val = 1.0 - P / P_max_c
        return np.maximum(val, 0.0)     # clip at 0 (no negative growth)

# ── pH range ──────────────────────────────────────────────────────────────────
pH_range = np.linspace(3.0, 8.0, 1000)

def f_HA_of_pH(pH_val):
    return 1.0 / (1.0 + 10.0**(pH_val - pKa))

def P_of_pH(pH_val, p_total):
    """Undissociated acid concentration [mM] as a function of pH."""
    return f_HA_of_pH(pH_val) * (p_total / Mw) * 1000.0

# ── Compute µ_rel vs pH for 1 g/L and 10 g/L ──────────────────────────────────
p_concentrations = [1.0, 10.0]   # g/L
colors = ['steelblue', 'tomato']

fig, ax = plt.subplots(figsize=(8, 5))

pH_stop_list = []

for p_tot, col in zip(p_concentrations, colors):
    P_pH = P_of_pH(pH_range, p_tot)
    mu_rel_pH = mu_rel_of_P(P_pH)

    ax.plot(pH_range, mu_rel_pH, color=col, lw=2.5,
            label=f'$p_{{\\rm total}}$ = {p_tot} g/L')

    # Find pH where growth stops (µ_rel ≤ 0)
    # For Model 1 (hyperbolic) µ_rel → 0 asymptotically; growth never truly stops.
    # We define a threshold of µ_rel < 0.001 (< 0.1% of µ_max)
    if best_model == 1:
        threshold = 0.001
        idx_stop = np.where(mu_rel_pH < threshold)[0]
    else:
        idx_stop = np.where(mu_rel_pH <= 0)[0]

    if len(idx_stop) > 0:
        pH_stop = pH_range[idx_stop[0]]
        pH_stop_list.append(pH_stop)
        ax.axvline(pH_stop, color=col, linestyle=':', lw=1.5,
                   label=f'Growth stops at pH ≈ {pH_stop:.2f} ({p_tot} g/L)')
        print(f"p_total = {p_tot:4.0f} g/L → growth stops at pH ≈ {pH_stop:.3f}")
    else:
        print(f"p_total = {p_tot:4.0f} g/L → growth does not stop in pH range [3, 8]")

ax.axhline(0, color='k', lw=0.8, linestyle='-')
ax.set_xlabel('pH')
ax.set_ylabel(r'Relative growth rate  $\mu / \mu_0$  (–)')
ax.set_title(f'Part (d) – Growth rate vs pH  (Model {best_model})')
ax.set_xlim(3, 8)
ax.set_ylim(-0.05, 1.1)
ax.legend(fontsize=10)
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('part_d_mu_vs_pH.png')
plt.show()

In [ ]:
# ── Summary of all key results ─────────────────────────────────────────────────
print("=" * 55)
print("SUMMARY OF RESULTS – EXERCISE 2")
print("=" * 55)
print(f"\n(a)  f_HA at pH {pH}  = {f_HA:.6f}")
print("     P values (mM):", np.round(P_mM, 4))
print(f"\n(b)  µ_max (Model 1)  = {mu_max_b:.4f} h⁻¹")
print(f"     K_I  (Model 1)  = {KI_b:.4f} mM")
print(f"\n(c)  µ_max (Model 2)  = {mu_max_c:.4f} h⁻¹")
print(f"     P_max (Model 2)  = {P_max_c:.4f} mM")
print(f"\n     Model 1 R² = {r2_m1:.5f},  RMSE = {rmse_m1:.5f} h⁻¹")
print(f"     Model 2 R² = {r2_m2:.5f},  RMSE = {rmse_m2:.5f} h⁻¹")
print(f"\n     → Best model: Model {best_model}")
print(f"\n(d)  Growth stops (pH, p_total):")
for p_tot, pH_stop in zip(p_concentrations, pH_stop_list):
    print(f"       {p_tot} g/L  →  pH ≈ {pH_stop:.3f}")